# Seção 4 – Análise Exploratória dos Dados (EDA)

Nesta seção, realizo a Análise Exploratória dos Dados (EDA) com o objetivo de entender
a estrutura dos datasets, identificar padrões iniciais, possíveis inconsistências,
valores ausentes e obter insights preliminares que apoiarão análises futuras.

Os dados utilizados simulam informações de clientes, serviços e contratos relacionados
a churn (cancelamento), cenário muito comum em análises de negócio.


In [ ]:
import pandas as pd

## Importação e visão geral dos dados

Nesta etapa inicial, o objetivo é conhecer os dados com os quais iremos trabalhar.
Aqui buscamos entender:
- quais informações estão disponíveis
- quantidade de registros
- tipos de variáveis
- possíveis problemas iniciais nos dados


In [ ]:
df_clientes = pd.read_csv("files/churn_customers.csv")
df_clientes

In [ ]:
df_clientes.head(8)

In [ ]:
df_clientes.tail(7)

In [ ]:
df_clientes.info()

In [ ]:
df_clientes.describe()

In [ ]:
df_servicos = pd.read_csv("files/churn_services.csv")

In [ ]:
df_servicos

In [ ]:
df_servicos.describe()

In [ ]:
df_contratos = pd.read_csv("files/churn_contracts.csv")

In [ ]:
df_contratos

In [ ]:
df_contratos.describe()

### Primeiras observações

A partir da visualização inicial, é possível identificar a presença de variáveis
numéricas e categóricas. Também observamos que algumas colunas podem exigir
tratamento posterior, como ajustes de tipo ou valores ausentes.


## Transformação e Padronização dos Dados

Nesta etapa, realizamos ajustes nos dados para garantir consistência,
facilidade de análise e correta tipagem das variáveis.

As transformações incluem:
- conversão de variáveis numéricas
- tratamento de valores inválidos
- padronização e renomeação de colunas


In [ ]:
# Verificando os tipos de dados do DataFrame de contratos
df_contratos.dtypes

In [ ]:
# Convertendo a coluna TotalCharges para numérico
# Valores inválidos são convertidos para NaN
df_contratos["TotalCharges"] = pd.to_numeric(df_contratos["TotalCharges"], errors="coerce")

In [ ]:
df_contratos.dtypes

In [ ]:
# Conferindo o resultado após a conversão
df_contratos.info()

### Padronização dos dados de clientes

Para melhorar a legibilidade e facilitar análises futuras, realizamos a
renomeação de colunas, tornando os nomes mais descritivos e alinhados ao
contexto de negócio.


In [ ]:
df_clientes.info()

In [ ]:
df_clientes = df_clientes.rename(columns={"SeniorCitizen": "ElderlyPerson"})

In [ ]:
df_clientes.info()

In [ ]:
df_clientes.rename(columns={"Partner": "Casado(a)"}, inplace=True)

In [ ]:
df_clientes.info()

In [ ]:
# Renomeando colunas para nomes mais descritivos
df_clientes.rename(columns={
    "customerID": "IDCliente",
    "gender": "Genero",
    "SeniorCitizen": "PessoaIdosa",
    "Partner": "Casado(a)",
    "Dependents": "Dependentes"
}, inplace=True)

In [ ]:
# Verificando a estrutura após as alterações
df_clientes.info()

## União dos DataFrames

Nesta etapa, realizamos a junção dos dados de clientes, serviços e contratos,
criando uma base única para análise de churn.

A união é feita utilizando o identificador do cliente como chave principal.


In [ ]:
len(df_servicos)

In [ ]:
df_servicos.head()

In [ ]:
# Padronizando o nome da chave no DataFrame de serviços
df_servicos.rename(columns={"customerID":"IDCliente"}, inplace=True)

In [ ]:
df_servicos.info()

In [ ]:
# União entre clientes e serviços
df_temp = df_clientes.merge(df_servicos, on=["IDCliente"])

In [ ]:
df_temp.info()

In [ ]:
# União com o DataFrame de contratos
df_churn_temp = df_temp.merge(df_contratos, left_on=["IDCliente"], right_on=["customerID"])

In [ ]:
df_churn_temp.info()

In [ ]:
df_churn = df_clientes.merge(df_servicos, on=["IDCliente"]).merge(df_contratos, left_on=["IDCliente"], right_on=["customerID"])

In [ ]:
df_churn

In [ ]:
# Removendo coluna duplicada após o merge
df_churn.drop(["customerID"], inplace=True, axis=1) # axis=1 é relacionado a coluna

In [ ]:
# Visualizando a estrutura final
df_churn.info()

## Tratamento de Valores Ausentes

Nesta etapa da Análise Exploratória dos Dados, o objetivo é identificar e tratar
valores ausentes (missing values), que podem impactar diretamente análises
estatísticas, visualizações e modelos futuros.

Valores ausentes são comuns em bases reais e podem ocorrer por falhas de
coleta, inconsistências operacionais ou ausência de informação no momento
do registro.


In [ ]:
df_churn.isna().sum()

In [ ]:
# Exibe os registros que possuem pelo menos um valor ausente
df_churn[df_churn.isna().any(axis=1)]

In [ ]:
# Quantidade de colunas que possuem ao menos um valor ausente
df_churn.isna().any(axis=0).sum()

### Detecção de valores ausentes

Inicialmente, realizamos a identificação de valores ausentes no dataset,
verificando:

- a quantidade total de valores ausentes por coluna
- quais registros possuem pelo menos um valor ausente
- quantas colunas apresentam algum tipo de ausência de dados


### Estratégias de tratamento de valores ausentes

Após identificar os valores ausentes, avaliamos diferentes estratégias de
tratamento, como:

- remoção de colunas
- remoção de linhas
- imputação de valores (substituição)

A escolha da estratégia depende do contexto do negócio e do impacto da
variável na análise.


In [ ]:
# Remove colunas que possuem qualquer valor ausente
# (estratégia não recomendada na maioria dos casos)
df_churn.dropna(axis=1)

In [ ]:
# Remove colunas apenas se TODOS os valores forem ausentes
df_churn.dropna(axis=1, how="all")

In [ ]:
# Remove linhas que possuem valores ausentes
df_churn.dropna(axis=0)

### Imputação de Valores Ausentes

In [ ]:
# Preenche todos os valores ausentes com zero
df_churn.fillna(0)

In [ ]:
# Preenche valores ausentes apenas da coluna TotalCharges com zero
df_churn.fillna(value={"TotalCharges":0})

In [ ]:
# Preenche valores ausentes da coluna TotalCharges com a média da própria coluna
df_churn.fillna({"TotalCharges":df_churn["TotalCharges"].mean()})

## Distribuição das Variáveis Categóricas

Nesta etapa da Análise Exploratória dos Dados, analisamos a distribuição das
variáveis categóricas com o objetivo de entender como os dados estão
organizados entre diferentes categorias.

Essa análise permite:
- identificar classes mais frequentes
- avaliar possíveis desbalanceamentos
- obter insights iniciais sobre o comportamento dos clientes em relação ao churn


In [ ]:
df_churn["Churn"].unique()

In [ ]:
df_churn["Churn"].value_counts()

In [ ]:
df_churn["Churn"].value_counts(normalize=True)

In [ ]:
df_churn["Churn"].value_counts().plot.bar()

In [ ]:
ax = df_churn["Churn"].value_counts(normalize=True).plot.bar()
ax.bar_label(ax.containers[0])

### Distribuição de outras variáveis categóricas

Também analisamos variáveis como:
- tipo de contrato
- gênero do cliente

Essas variáveis ajudam a identificar padrões e possíveis relações com o churn,
que poderão ser aprofundadas em análises posteriores.


In [ ]:
df_churn["Contract"].unique()

In [ ]:
df_churn["Contract"].value_counts()

In [ ]:
ax = df_churn["Contract"].value_counts().plot.bar()
ax.bar_label(ax.containers[0])

In [ ]:
df_churn["Genero"].value_counts()

In [ ]:
ax = df_churn["Genero"].value_counts().plot.bar()
ax.bar_label(ax.containers[0])

## Distribuição das Variáveis Numéricas

Nesta etapa, analisamos a distribuição das variáveis numéricas para entender
o comportamento dos dados ao longo do tempo e seus valores monetários.

A análise inclui:
- visualização da distribuição
- medidas estatísticas descritivas
- avaliação da dispersão dos dados


### Análise da variável tenure

A variável **tenure** representa o tempo de permanência do cliente,
em meses, na base.

A análise dessa variável é importante para entender o comportamento de churn
em relação ao tempo de contrato.


In [ ]:
df_churn["tenure"].value_counts()

In [ ]:
df_churn["tenure"].plot.hist()

**Observação:**  
Há uma grande concentração de clientes nos extremos da distribuição,
ou seja, contratos com poucos meses de duração e contratos longos,
acima de aproximadamente 65 meses.


### Distribuição da variável MonthlyCharges

In [ ]:
# Histograma dos valores de cobrança mensal
df_churn["MonthlyCharges"].plot.hist()

### Medidas estatísticas da variável tenure

In [ ]:
df_churn["tenure"].mean()

In [ ]:
df_churn["tenure"].median()

In [ ]:
df_churn["tenure"].mode()

In [ ]:
# Variação de 24 meses em relação a média
df_churn["tenure"].std()

In [ ]:
# Coeficiente de variação para avaliar a dispersão relativa dos dados
df_churn["tenure"].std() / df_churn["tenure"].mean() * 100

## Filtros e Agrupamento de Valores

Nesta etapa, aplicamos filtros e agrupamentos para entender o comportamento
dos clientes ao longo do tempo de contrato (tenure), buscando padrões
associados ao churn.


In [ ]:
len(df_churn[df_churn["tenure"] == 1])

In [ ]:
len(df_churn[df_churn["tenure"] == 1]) / len(df_churn) * 100

In [ ]:
len(df_churn[(df_churn["tenure"] >= 1) & (df_churn["tenure"] <= 5)])

In [ ]:
len(df_churn[(df_churn["tenure"] >= 1) & (df_churn["tenure"] <= 12)]) / len(df_churn) * 100

In [ ]:
len(df_churn[(df_churn["Genero"] == "Male") & (df_churn["tenure"] <= 5)])

In [ ]:
df_churn.groupby(["tenure"])["tenure"].count().sort_values(ascending=False)

In [ ]:
df_churn.groupby(["tenure"])["tenure"].count().sort_values().plot.barh(figsize=(17,17))

### Interpretação dos resultados

Observa-se uma concentração relevante de clientes nos primeiros meses
de contrato, indicando maior vulnerabilidade ao churn nesse período inicial.
Esse comportamento sugere que estratégias de retenção devem ser priorizadas
nos primeiros meses da jornada do cliente.


## Tabelas de Contingência

Nesta análise, cruzamos o tipo de contrato com a variável churn para identificar
relações entre modelo contratual e cancelamento.


In [ ]:
pd.crosstab(df_churn["Churn"], df_churn["Contract"])

In [ ]:
pd.crosstab(df_churn["Churn"], df_churn["Contract"], normalize=True, margins=True, margins_name="Total")

In [ ]:
pd.crosstab(df_churn["Churn"], df_churn["Contract"], normalize="index", margins=True, margins_name="Total")

### Principais insights

- Clientes com contrato mensal representam a maior parte da base.
- Entre os clientes que cancelaram, a predominância é de contratos mensais.
- Isso indica uma forte associação entre contratos de curto prazo e churn,
sugerindo que contratos mais longos tendem a aumentar a retenção.


## Teste do Qui-Quadrado
- HO (Hipótese Nula): AS duas variávies são independentes.
- H1 (Hipótese Complementar): As duas variávies não são independentes.
- No teste, o objetivo é confirmar ou recusar a hipótese nula.
- Quando o valor de probabilidade H0 for menor que 0.05 (p-value) recusamos a hipótese nula e aceitamos a complementar.

In [ ]:
df_tabcont = pd.crosstab(df_churn["Churn"], df_churn["Contract"])

In [ ]:
from scipy.stats import chi2_contingency

In [ ]:
chi_scores = chi2_contingency(df_tabcont)

In [ ]:
chi_scores

In [ ]:
# Número decimal de uma notação científica
pd.set_option("display.float_format", lambda x: "%.15f" % x)

In [ ]:
scores_churn = pd.Series(chi_scores[0])
pvalues_churn = pd.Series(chi_scores[1])

In [ ]:
df_chi_churn = pd.DataFrame({
    "Qui2": scores_churn,
    "PValue": pvalues_churn
})

In [ ]:
df_chi_churn

As variáveis não são independentes. E pelo chi² alto, podemos afirmar que há uma alta correlação entre as variáves.

## Correlação entre variável qualitativa e quantitativa

Hipótese: cliente com menos de 6 meses de contrato é mais propenso ao churn.

In [ ]:
import numpy as np

In [ ]:
df_churn["TempoMenor6Meses"] = np.where(df_churn["tenure"] < 6, "Sim", "Não")

In [ ]:
df_churn.head()

In [ ]:
df_crosstab_tempo = pd.crosstab(df_churn["Churn"], df_churn["TempoMenor6Meses"])

In [ ]:
df_crosstab_tempo

In [ ]:
chi_scores = chi2_contingency(df_crosstab_tempo)

In [ ]:
chi_scores

In [ ]:
scores_churn = pd.Series(chi_scores[0])
pvalues_churn = pd.Series(chi_scores[1])

In [ ]:
df_chi_churn = pd.DataFrame({
    "Qui2": scores_churn,
    "PValue": pvalues_churn
})

In [ ]:
df_chi_churn

Os resultados indicam dependência estatística entre churn e tempo de contrato inferior a 6 meses, embora a força da associação seja menor quando comparada ao tipo de contrato.

## Correlação 2 variáveis numéricas

In [ ]:
# Correlação com Pearson
# Quanto mais tempo de contrato maior o valor pago.
df_churn["tenure"].corr(df_churn["TotalCharges"])

In [ ]:
df_churn["tenure"].corr(df_churn["TotalCharges"], method="spearman")

In [ ]:
df_churn.plot.scatter(x="tenure", y="TotalCharges")